# Calculadoras de Distribuições Contínuas

Este notebook contém calculadoras interativas para as principais distribuições de probabilidade contínuas.

## 💡 Aproximações Possíveis

| De (Original) | Para (Aproximada) | Critério |
| :--- | :--- | :--- |
| **t-Student** | **Normal** | $GL \to \infty$ (tipicamente $GL \geq 30$) |

## 1. Distribuição Normal

A distribuição normal (ou gaussiana) é a mais importante distribuição contínua.

**Parâmetros:**
- μ (mu): média
- σ (sigma): desvio padrão
- X: valor da variável

In [ ]:
import scipy.stats as stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')

# Flag para prevenir chamadas duplicadas
_updating_normal = False

# Criar widgets
modo = widgets.ToggleButtons(
    options=['Direto (x → P)', 'Inverso (P → x)'],
    value='Direto (x → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

mu_widget = widgets.FloatText(value=0, description='μ (média):', style={'description_width': 'initial'})
sigma_widget = widgets.FloatText(value=1, description='σ (desvio padrão):', style={'description_width': 'initial'})
x_widget = widgets.FloatText(value=1.96, description='x (valor):', style={'description_width': 'initial'})
a_widget = widgets.FloatText(value=-1.96, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget = widgets.FloatText(value=1.96, description='b (limite sup.):', style={'description_width': 'initial'})
prob_widget = widgets.FloatText(value=0.975, description='Probabilidade:', style={'description_width': 'initial'})

calc_type = widgets.Dropdown(
    options=[
        ('P(X ≤ x)', 'cdf'), 
        ('P(X ≥ x)', 'sf'), 
        ('P(X > x)', 'gt'), 
        ('P(X < x)', 'lt'), 
        ('P(a ≤ X ≤ b)', 'interval'),
        ('f(x) (PDF)', 'pdf')
    ],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_inv = widgets.Dropdown(
    options=[('P(X ≤ x) = p', 'cdf'), ('P(X ≥ x) = p', 'sf')],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph = widgets.Checkbox(value=True, description='Mostrar gráfico')

output = widgets.Output()
input_container = widgets.VBox()

def calcular_normal(change=None):
    global _updating_normal
    if _updating_normal:
        return
    _updating_normal = True
    
    try:
        with output:
            clear_output(wait=True)
            try:
                mu = mu_widget.value
                sigma = sigma_widget.value
                
                if sigma <= 0:
                    print("❌ σ deve ser maior que 0")
                    return
                
                dist = stats.norm(mu, sigma)
                
                # Indicador visual do modo
                if modo.value == 'Direto (x → P)':
                    print("📍 MODO: Direto (dado x, calcular probabilidade)")
                else:
                    print("📍 MODO: Inverso (dada probabilidade, calcular x)")
                print("="*60 + "\n")
                
                # Modo Direto: x -> probabilidade
                if modo.value == 'Direto (x → P)':
                    if calc_type.value == 'interval':
                        # Modo Intervalo
                        a = a_widget.value
                        b = b_widget.value
                        
                        if a >= b:
                            print("❌ a deve ser menor que b")
                            return
                        
                        prob = dist.cdf(b) - dist.cdf(a)
                        print(f"P({a:.4f} ≤ X ≤ {b:.4f}) = {prob:.6f}")
                        print(f"\nLimite inferior: a = {a:.4f} (Z = {(a-mu)/sigma:.4f})")
                        print(f"Limite superior: b = {b:.4f} (Z = {(b-mu)/sigma:.4f})")
                        print(f"Amplitude: {b-a:.4f}")
                        
                        if show_graph.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, 'b-', linewidth=2, label='PDF')
                            ax.axvline(a, color='red', linestyle='--', linewidth=2, label=f'a = {a:.2f}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b:.2f}')
                            ax.axvline(mu, color='green', linestyle='--', linewidth=2, label=f'μ = {mu:.2f}')
                            
                            x_fill = x_vals[(x_vals >= a) & (x_vals <= b)]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='blue', label=f'P = {prob:.4f}')
                            
                            ax.set_xlabel('x', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Normal: μ={mu}, σ={sigma}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        # Modo Direto normal
                        x = x_widget.value
                        z = (x - mu) / sigma
                        
                        if calc_type.value == 'pdf':
                            result = dist.pdf(x)
                            print(f"f({x:.4f}) = {result:.6f}")
                        elif calc_type.value == 'cdf':
                            result = dist.cdf(x)
                            print(f"P(X ≤ {x:.4f}) = {result:.6f}")
                        elif calc_type.value == 'sf':
                            result = dist.sf(x)
                            print(f"P(X ≥ {x:.4f}) = {result:.6f}")
                        elif calc_type.value == 'gt':
                            result = dist.sf(x)
                            print(f"P(X > {x:.4f}) = {result:.6f}")
                        elif calc_type.value == 'lt':
                            result = dist.cdf(x)
                            print(f"P(X < {x:.4f}) = {result:.6f}")
                        
                        print(f"\nZ-score = {z:.4f}")
                        print(f"Média μ = {dist.mean():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        print(f"Variância σ² = {dist.var():.4f}")
                        
                        if show_graph.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, 'b-', linewidth=2, label='PDF')
                            ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f}')
                            ax.axvline(mu, color='green', linestyle='--', linewidth=2, label=f'μ = {mu:.2f}')
                            
                            if calc_type.value in ['cdf', 'lt']:
                                x_fill = x_vals[x_vals <= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='blue')
                            elif calc_type.value in ['sf', 'gt']:
                                x_fill = x_vals[x_vals >= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='blue')
                            
                            ax.set_xlabel('x', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Normal: μ={mu}, σ={sigma}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso: probabilidade -> x
                else:
                    p = prob_widget.value
                    
                    if p <= 0 or p >= 1:
                        print("❌ Probabilidade deve estar entre 0 e 1 (exclusivo)")
                        return
                    
                    if calc_type_inv.value == 'cdf':
                        x = dist.ppf(p)
                        print(f"Para P(X ≤ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    else:  # sf
                        x = dist.isf(p)
                        print(f"Para P(X ≥ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    
                    z = (x - mu) / sigma
                    print(f"\nZ-score = {z:.4f}")
                    print(f"Média μ = {dist.mean():.4f}")
                    print(f"Desvio Padrão σ = {dist.std():.4f}")
                    
                    if show_graph.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
                        pdf_vals = dist.pdf(x_vals)
                        
                        ax.plot(x_vals, pdf_vals, 'b-', linewidth=2, label='PDF')
                        ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f} (quantil)')
                        ax.axvline(mu, color='green', linestyle='--', linewidth=2, label=f'μ = {mu:.2f}')
                        
                        if calc_type_inv.value == 'cdf':
                            x_fill = x_vals[x_vals <= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='blue', label=f'Área = {p:.4f}')
                        else:
                            x_fill = x_vals[x_vals >= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='blue', label=f'Área = {p:.4f}')
                        
                        ax.set_xlabel('x', fontsize=12)
                        ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                        ax.set_title(f'Distribuição Normal: μ={mu}, σ={sigma}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_normal = False

def atualizar_interface(change=None):
    global _updating_normal
    if _updating_normal:
        return
    _updating_normal = True
    
    try:
        if modo.value == 'Direto (x → P)':
            if calc_type.value == 'interval':
                input_container.children = [calc_type, a_widget, b_widget]
            else:
                input_container.children = [calc_type, x_widget]
        else:
            input_container.children = [calc_type_inv, prob_widget]
        calcular_normal()
    finally:
        _updating_normal = False

# Registrar observadores apenas uma vez
if not hasattr(modo, '_handler_registered'):
    modo.observe(atualizar_interface, 'value')
    calc_type.observe(atualizar_interface, 'value')
    calc_type_inv.observe(atualizar_interface, 'value')
    mu_widget.observe(calcular_normal, 'value')
    sigma_widget.observe(calcular_normal, 'value')
    x_widget.observe(calcular_normal, 'value')
    a_widget.observe(calcular_normal, 'value')
    b_widget.observe(calcular_normal, 'value')
    prob_widget.observe(calcular_normal, 'value')
    show_graph.observe(calcular_normal, 'value')
    
    # Marcar como registrados
    modo._handler_registered = True
    calc_type._handler_registered = True
    calc_type_inv._handler_registered = True
    mu_widget._handler_registered = True
    sigma_widget._handler_registered = True
    x_widget._handler_registered = True
    a_widget._handler_registered = True
    b_widget._handler_registered = True
    prob_widget._handler_registered = True
    show_graph._handler_registered = True

# Mostrar interface
display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Normal</h3>"),
    modo,
    mu_widget, 
    sigma_widget, 
    input_container,
    show_graph, 
    output
]))

# Inicializar interface
atualizar_interface()

## 2. Distribuição t de Student

A distribuição t é usada quando o tamanho da amostra é pequeno e o desvio padrão populacional é desconhecido.

**Parâmetros:**
- df (GL): graus de liberdade
- X: valor da variável

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_t = False

# Criar widgets t-Student
modo_t = widgets.ToggleButtons(
    options=['Direto (t → P)', 'Inverso (P → t)'],
    value='Direto (t → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

df_t = widgets.IntText(value=10, description='df (graus lib.):', style={'description_width': 'initial'})
x_t = widgets.FloatText(value=2.0, description='t (valor):', style={'description_width': 'initial'})
a_t = widgets.FloatText(value=-2.0, description='a (limite inf.):', style={'description_width': 'initial'})
b_t = widgets.FloatText(value=2.0, description='b (limite sup.):', style={'description_width': 'initial'})
prob_t = widgets.FloatText(value=0.975, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_t = widgets.Dropdown(
    options=[
        ('P(T ≤ t)', 'cdf'), 
        ('P(T ≥ t)', 'sf'), 
        ('P(|T| ≥ |t|) bilateral', 'two_tailed'), 
        ('P(a ≤ T ≤ b)', 'interval'),
        ('f(t) (PDF)', 'pdf')
    ],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_t_inv = widgets.Dropdown(
    options=[('P(T ≤ t) = p', 'cdf'), ('P(T ≥ t) = p', 'sf')],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph_t = widgets.Checkbox(value=True, description='Mostrar gráfico')
output_t = widgets.Output()
input_container_t = widgets.VBox()

def calcular_t(change=None):
    global _updating_t
    if _updating_t:
        return
    _updating_t = True
    
    try:
        with output_t:
            clear_output(wait=True)
            try:
                df = df_t.value
                
                if df < 1:
                    print("❌ df deve ser ≥ 1")
                    return
                
                dist = stats.t(df)
                
                # Modo Direto
                if modo_t.value == 'Direto (t → P)':
                    if calc_type_t.value == 'interval':
                        a = a_t.value
                        b = b_t.value
                        
                        if a >= b:
                            print("❌ a deve ser menor que b")
                            return
                        
                        prob = dist.cdf(b) - dist.cdf(a)
                        print(f"P({a:.4f} ≤ T ≤ {b:.4f}) = {prob:.6f}")
                        print(f"\nLimite inferior: a = {a:.4f}")
                        print(f"Limite superior: b = {b:.4f}")
                        print(f"Amplitude: {b-a:.4f}")
                        
                        if show_graph_t.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(-5, 5, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, 'r-', linewidth=2, label=f't (df={df})')
                            ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a:.2f}')
                            ax.axvline(b, color='red', linestyle='--', linewidth=2, label=f'b = {b:.2f}')
                            ax.axvline(0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
                            
                            x_fill = x_vals[(x_vals >= a) & (x_vals <= b)]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='red', label=f'P = {prob:.4f}')
                            
                            ax.set_xlabel('t', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(t)', fontsize=12)
                            ax.set_title(f'Distribuição t de Student: df={df}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        x = x_t.value
                        
                        if calc_type_t.value == 'pdf':
                            result = dist.pdf(x)
                            print(f"f({x:.4f}) = {result:.6f}")
                        elif calc_type_t.value == 'cdf':
                            result = dist.cdf(x)
                            print(f"P(T ≤ {x:.4f}) = {result:.6f}")
                        elif calc_type_t.value == 'sf':
                            result = dist.sf(x)
                            print(f"P(T ≥ {x:.4f}) = {result:.6f}")
                        elif calc_type_t.value == 'two_tailed':
                            result = 2 * dist.sf(abs(x))
                            print(f"P(|T| ≥ {abs(x):.4f}) = {result:.6f}")
                            print(f"(teste bilateral)")
                        
                        print(f"\nGraus de liberdade = {df}")
                        print(f"Média = {dist.mean():.4f}" if df > 1 else "\nMédia = indefinida (df ≤ 1)")
                        if df > 2:
                            print(f"Variância = {dist.var():.4f}")
                        
                        # Aproximação
                        print("\n" + "="*50)
                        print("🔔 AVISOS DE APROXIMAÇÃO:")
                        if df >= 30:
                            print(f"\n✅ Pode aproximar para NORMAL PADRÃO N(0,1)")
                            print(f"   Critério: GL = {df} ≥ 30")
                        else:
                            print(f"\n⚠️  GL = {df} < 30. Use a distribuição t de Student.")
                        
                        if show_graph_t.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(-5, 5, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, 'r-', linewidth=2, label=f't (df={df})')
                            if df >= 30:
                                normal_pdf = stats.norm(0, 1).pdf(x_vals)
                                ax.plot(x_vals, normal_pdf, 'b--', linewidth=2, alpha=0.6, label='Normal(0,1)')
                            
                            ax.axvline(x, color='darkred', linestyle='--', linewidth=2, label=f't = {x:.2f}')
                            ax.axvline(0, color='gray', linestyle='-', linewidth=1, alpha=0.5)
                            
                            if calc_type_t.value == 'cdf':
                                x_fill = x_vals[x_vals <= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='red')
                            elif calc_type_t.value == 'sf':
                                x_fill = x_vals[x_vals >= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='red')
                            elif calc_type_t.value == 'two_tailed':
                                x_fill_left = x_vals[x_vals <= -abs(x)]
                                x_fill_right = x_vals[x_vals >= abs(x)]
                                ax.fill_between(x_fill_left, dist.pdf(x_fill_left), alpha=0.3, color='red')
                                ax.fill_between(x_fill_right, dist.pdf(x_fill_right), alpha=0.3, color='red')
                            
                            ax.set_xlabel('t', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(t)', fontsize=12)
                            ax.set_title(f'Distribuição t de Student: df={df}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    p = prob_t.value
                    
                    if p <= 0 or p >= 1:
                        print("❌ Probabilidade deve estar entre 0 e 1 (exclusivo)")
                        return
                    
                    if calc_type_t_inv.value == 'cdf':
                        x = dist.ppf(p)
                        print(f"Para P(T ≤ t) = {p:.6f}")
                        print(f"t = {x:.4f}")
                    else:  # sf
                        x = dist.isf(p)
                        print(f"Para P(T ≥ t) = {p:.6f}")
                        print(f"t = {x:.4f}")
                    
                    print(f"\nGraus de liberdade = {df}")
                    
                    if show_graph_t.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.linspace(-5, 5, 1000)
                        pdf_vals = dist.pdf(x_vals)
                        
                        ax.plot(x_vals, pdf_vals, 'r-', linewidth=2, label=f't (df={df})')
                        ax.axvline(x, color='darkred', linestyle='--', linewidth=2, label=f't = {x:.2f} (quantil)')
                        
                        if calc_type_t_inv.value == 'cdf':
                            x_fill = x_vals[x_vals <= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='red', label=f'Área = {p:.4f}')
                        else:
                            x_fill = x_vals[x_vals >= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='red', label=f'Área = {p:.4f}')
                        
                        ax.set_xlabel('t', fontsize=12)
                        ax.set_ylabel('Densidade de Probabilidade f(t)', fontsize=12)
                        ax.set_title(f'Distribuição t de Student: df={df}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_t = False

def atualizar_interface_t(change=None):
    global _updating_t
    if _updating_t:
        return
    _updating_t = True
    
    try:
        if modo_t.value == 'Direto (t → P)':
            if calc_type_t.value == 'interval':
                input_container_t.children = [calc_type_t, a_t, b_t]
            else:
                input_container_t.children = [calc_type_t, x_t]
        else:
            input_container_t.children = [calc_type_t_inv, prob_t]
        calcular_t()
    finally:
        _updating_t = False

# Registrar observadores apenas uma vez
if not hasattr(modo_t, '_handler_registered'):
    modo_t.observe(atualizar_interface_t, 'value')
    calc_type_t.observe(atualizar_interface_t, 'value')
    df_t.observe(calcular_t, 'value')
    x_t.observe(calcular_t, 'value')
    a_t.observe(calcular_t, 'value')
    b_t.observe(calcular_t, 'value')
    prob_t.observe(calcular_t, 'value')
    calc_type_t_inv.observe(calcular_t, 'value')
    show_graph_t.observe(calcular_t, 'value')
    
    # Marcar como registrados
    modo_t._handler_registered = True
    calc_type_t._handler_registered = True
    df_t._handler_registered = True
    x_t._handler_registered = True
    a_t._handler_registered = True
    b_t._handler_registered = True
    prob_t._handler_registered = True
    calc_type_t_inv._handler_registered = True
    show_graph_t._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora t de Student</h3>"),
    modo_t,
    df_t, 
    input_container_t,
    show_graph_t, 
    output_t
]))

atualizar_interface_t()

## 3. Distribuição Qui-Quadrado (χ²)

A distribuição qui-quadrado é usada em testes de hipóteses e intervalos de confiança para variâncias.

**Parâmetros:**
- df (GL): graus de liberdade
- X: valor da variável

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_chi = False

# Criar widgets
modo_chi = widgets.ToggleButtons(
    options=['Direto (x → P)', 'Inverso (P → x)'],
    value='Direto (x → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

df_chi = widgets.IntText(value=5, description='df (graus lib.):', style={'description_width': 'initial'})
x_chi = widgets.FloatText(value=7.0, description='χ² (valor):', style={'description_width': 'initial'})
a_chi = widgets.FloatText(value=2.0, description='a (limite inf.):', style={'description_width': 'initial'})
b_chi = widgets.FloatText(value=10.0, description='b (limite sup.):', style={'description_width': 'initial'})
prob_chi = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_chi = widgets.Dropdown(
    options=[
        ('P(χ² ≤ x)', 'cdf'), 
        ('P(χ² ≥ x)', 'sf'), 
        ('P(a ≤ χ² ≤ b)', 'interval'),
        ('f(x) (PDF)', 'pdf')
    ],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_chi_inv = widgets.Dropdown(
    options=[('P(χ² ≤ x) = p', 'cdf'), ('P(χ² ≥ x) = p', 'sf')],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph_chi = widgets.Checkbox(value=True, description='Mostrar gráfico')
output_chi = widgets.Output()
input_container_chi = widgets.VBox()

def calcular_chi(change=None):
    global _updating_chi
    if _updating_chi:
        return
    _updating_chi = True
    
    try:
        with output_chi:
            clear_output(wait=True)
            try:
                df = df_chi.value
                
                if df < 1:
                    print("❌ df deve ser ≥ 1")
                    return
                
                dist = stats.chi2(df)
                
                # Modo Direto
                if modo_chi.value == 'Direto (x → P)':
                    if calc_type_chi.value == 'interval':
                        a = a_chi.value
                        b = b_chi.value
                        
                        if a < 0:
                            print("❌ a deve ser ≥ 0")
                            return
                        
                        if a >= b:
                            print("❌ a deve ser menor que b")
                            return
                        
                        prob = dist.cdf(b) - dist.cdf(a)
                        print(f"P({a:.4f} ≤ χ² ≤ {b:.4f}) = {prob:.6f}")
                        print(f"\nLimite inferior: a = {a:.4f}")
                        print(f"Limite superior: b = {b:.4f}")
                        print(f"Amplitude: {b-a:.4f}")
                        
                        if show_graph_chi.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(0, df + 4*np.sqrt(2*df), 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='darkorange', linewidth=2, label='PDF')
                            ax.axvline(a, color='red', linestyle='--', linewidth=2, label=f'a = {a:.2f}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b:.2f}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            x_fill = x_vals[(x_vals >= a) & (x_vals <= b)]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='orange', label=f'P = {prob:.4f}')
                            
                            ax.set_xlabel('χ²', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Qui-Quadrado: df={df}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        x = x_chi.value
                        
                        if x < 0:
                            print("❌ x deve ser ≥ 0")
                            return
                        
                        if calc_type_chi.value == 'pdf':
                            result = dist.pdf(x)
                            print(f"f({x:.4f}) = {result:.6f}")
                        elif calc_type_chi.value == 'cdf':
                            result = dist.cdf(x)
                            print(f"P(χ² ≤ {x:.4f}) = {result:.6f}")
                        elif calc_type_chi.value == 'sf':
                            result = dist.sf(x)
                            print(f"P(χ² ≥ {x:.4f}) = {result:.6f}")
                        
                        print(f"\nGraus de liberdade = {df}")
                        print(f"Média = {dist.mean():.4f}")
                        print(f"Variância = {dist.var():.4f}")
                        print(f"Desvio Padrão = {dist.std():.4f}")
                        
                        if show_graph_chi.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(0, df + 4*np.sqrt(2*df), 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='darkorange', linewidth=2, label='PDF')
                            ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            if calc_type_chi.value == 'cdf':
                                x_fill = x_vals[x_vals <= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='orange')
                            elif calc_type_chi.value == 'sf':
                                x_fill = x_vals[x_vals >= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='orange')
                            
                            ax.set_xlabel('χ²', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Qui-Quadrado: df={df}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    p = prob_chi.value
                    
                    if p <= 0 or p >= 1:
                        print("❌ Probabilidade deve estar entre 0 e 1 (exclusivo)")
                        return
                    
                    if calc_type_chi_inv.value == 'cdf':
                        x = dist.ppf(p)
                        print(f"Para P(χ² ≤ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    else:  # sf
                        x = dist.isf(p)
                        print(f"Para P(χ² ≥ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    
                    print(f"\nGraus de liberdade = {df}")
                    print(f"Média = {dist.mean():.4f}")
                    print(f"Variância = {dist.var():.4f}")
                    
                    if show_graph_chi.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.linspace(0, df + 4*np.sqrt(2*df), 1000)
                        pdf_vals = dist.pdf(x_vals)
                        
                        ax.plot(x_vals, pdf_vals, color='darkorange', linewidth=2, label='PDF')
                        ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f} (quantil)')
                        ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        if calc_type_chi_inv.value == 'cdf':
                            x_fill = x_vals[x_vals <= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='orange', label=f'Área = {p:.4f}')
                        else:
                            x_fill = x_vals[x_vals >= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='orange', label=f'Área = {p:.4f}')
                        
                        ax.set_xlabel('χ²', fontsize=12)
                        ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                        ax.set_title(f'Distribuição Qui-Quadrado: df={df}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_chi = False

def atualizar_interface_chi(change=None):
    global _updating_chi
    if _updating_chi:
        return
    _updating_chi = True
    
    try:
        if modo_chi.value == 'Direto (x → P)':
            if calc_type_chi.value == 'interval':
                input_container_chi.children = [calc_type_chi, a_chi, b_chi]
            else:
                input_container_chi.children = [calc_type_chi, x_chi]
        else:
            input_container_chi.children = [calc_type_chi_inv, prob_chi]
        calcular_chi()
    finally:
        _updating_chi = False

# Registrar observadores apenas uma vez
if not hasattr(modo_chi, '_handler_registered'):
    modo_chi.observe(atualizar_interface_chi, 'value')
    calc_type_chi.observe(atualizar_interface_chi, 'value')
    df_chi.observe(calcular_chi, 'value')
    x_chi.observe(calcular_chi, 'value')
    a_chi.observe(calcular_chi, 'value')
    b_chi.observe(calcular_chi, 'value')
    prob_chi.observe(calcular_chi, 'value')
    calc_type_chi_inv.observe(calcular_chi, 'value')
    show_graph_chi.observe(calcular_chi, 'value')
    
    modo_chi._handler_registered = True
    calc_type_chi._handler_registered = True
    df_chi._handler_registered = True
    x_chi._handler_registered = True
    a_chi._handler_registered = True
    b_chi._handler_registered = True
    prob_chi._handler_registered = True
    calc_type_chi_inv._handler_registered = True
    show_graph_chi._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Qui-Quadrado</h3>"),
    modo_chi,
    df_chi, 
    input_container_chi,
    show_graph_chi, 
    output_chi
]))

atualizar_interface_chi()

## 4. Distribuição F (Fisher-Snedecor)

A distribuição F é usada em análise de variância (ANOVA) e testes de razão de variâncias.

**Parâmetros:**
- dfn: graus de liberdade do numerador
- dfd: graus de liberdade do denominador
- X: valor da variável

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_f = False

# Criar widgets
modo_f = widgets.ToggleButtons(
    options=['Direto (x → P)', 'Inverso (P → x)'],
    value='Direto (x → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

dfn_f = widgets.IntText(value=5, description='dfn (num.):', style={'description_width': 'initial'})
dfd_f = widgets.IntText(value=10, description='dfd (denom.):', style={'description_width': 'initial'})
x_f = widgets.FloatText(value=2.5, description='F (valor):', style={'description_width': 'initial'})
a_f = widgets.FloatText(value=1.0, description='a (limite inf.):', style={'description_width': 'initial'})
b_f = widgets.FloatText(value=4.0, description='b (limite sup.):', style={'description_width': 'initial'})
prob_f = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_f = widgets.Dropdown(
    options=[
        ('P(F ≤ x)', 'cdf'), 
        ('P(F ≥ x)', 'sf'), 
        ('P(a ≤ F ≤ b)', 'interval'),
        ('f(x) (PDF)', 'pdf')
    ],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_f_inv = widgets.Dropdown(
    options=[('P(F ≤ x) = p', 'cdf'), ('P(F ≥ x) = p', 'sf')],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph_f = widgets.Checkbox(value=True, description='Mostrar gráfico')
output_f = widgets.Output()
input_container_f = widgets.VBox()

def calcular_f(change=None):
    global _updating_f
    if _updating_f:
        return
    _updating_f = True
    
    try:
        with output_f:
            clear_output(wait=True)
            try:
                dfn = dfn_f.value
                dfd = dfd_f.value
                
                if dfn < 1 or dfd < 1:
                    print("❌ dfn ≥ 1 e dfd ≥ 1")
                    return
                
                dist = stats.f(dfn, dfd)
                
                # Modo Direto
                if modo_f.value == 'Direto (x → P)':
                    if calc_type_f.value == 'interval':
                        a = a_f.value
                        b = b_f.value
                        
                        if a < 0:
                            print("❌ a deve ser ≥ 0")
                            return
                        
                        if a >= b:
                            print("❌ a deve ser menor que b")
                            return
                        
                        prob = dist.cdf(b) - dist.cdf(a)
                        print(f"P({a:.4f} ≤ F ≤ {b:.4f}) = {prob:.6f}")
                        print(f"\nLimite inferior: a = {a:.4f}")
                        print(f"Limite superior: b = {b:.4f}")
                        print(f"Amplitude: {b-a:.4f}")
                        
                        if show_graph_f.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(0.01, max(6, b+2), 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='purple', linewidth=2, label='PDF')
                            ax.axvline(a, color='red', linestyle='--', linewidth=2, label=f'a = {a:.2f}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b:.2f}')
                            if dfd > 2:
                                ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            x_fill = x_vals[(x_vals >= a) & (x_vals <= b)]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='purple', label=f'P = {prob:.4f}')
                            
                            ax.set_xlabel('F', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição F: dfn={dfn}, dfd={dfd}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        x = x_f.value
                        
                        if x < 0:
                            print("❌ x deve ser ≥ 0")
                            return
                        
                        if calc_type_f.value == 'pdf':
                            result = dist.pdf(x)
                            print(f"f({x:.4f}) = {result:.6f}")
                        elif calc_type_f.value == 'cdf':
                            result = dist.cdf(x)
                            print(f"P(F ≤ {x:.4f}) = {result:.6f}")
                        elif calc_type_f.value == 'sf':
                            result = dist.sf(x)
                            print(f"P(F ≥ {x:.4f}) = {result:.6f}")
                        
                        print(f"\nGraus de liberdade: dfn={dfn}, dfd={dfd}")
                        if dfd > 2:
                            print(f"Média = {dist.mean():.4f}")
                        if dfd > 4:
                            print(f"Variância = {dist.var():.4f}")
                        
                        if show_graph_f.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(0.01, max(6, x+2), 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='purple', linewidth=2, label='PDF')
                            ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'F = {x:.2f}')
                            if dfd > 2:
                                ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            if calc_type_f.value == 'cdf':
                                x_fill = x_vals[x_vals <= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='purple')
                            elif calc_type_f.value == 'sf':
                                x_fill = x_vals[x_vals >= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='purple')
                            
                            ax.set_xlabel('F', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição F: dfn={dfn}, dfd={dfd}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    p = prob_f.value
                    
                    if p <= 0 or p >= 1:
                        print("❌ Probabilidade deve estar entre 0 e 1 (exclusivo)")
                        return
                    
                    if calc_type_f_inv.value == 'cdf':
                        x = dist.ppf(p)
                        print(f"Para P(F ≤ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    else:  # sf
                        x = dist.isf(p)
                        print(f"Para P(F ≥ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    
                    print(f"\nGraus de liberdade: dfn={dfn}, dfd={dfd}")
                    if dfd > 2:
                        print(f"Média = {dist.mean():.4f}")
                    if dfd > 4:
                        print(f"Variância = {dist.var():.4f}")
                    
                    if show_graph_f.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.linspace(0.01, max(6, x+2), 1000)
                        pdf_vals = dist.pdf(x_vals)
                        
                        ax.plot(x_vals, pdf_vals, color='purple', linewidth=2, label='PDF')
                        ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'F = {x:.2f} (quantil)')
                        if dfd > 2:
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        if calc_type_f_inv.value == 'cdf':
                            x_fill = x_vals[x_vals <= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='purple', label=f'Área = {p:.4f}')
                        else:
                            x_fill = x_vals[x_vals >= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='purple', label=f'Área = {p:.4f}')
                        
                        ax.set_xlabel('F', fontsize=12)
                        ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                        ax.set_title(f'Distribuição F: dfn={dfn}, dfd={dfd}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_f = False

def atualizar_interface_f(change=None):
    global _updating_f
    if _updating_f:
        return
    _updating_f = True
    
    try:
        if modo_f.value == 'Direto (x → P)':
            if calc_type_f.value == 'interval':
                input_container_f.children = [calc_type_f, a_f, b_f]
            else:
                input_container_f.children = [calc_type_f, x_f]
        else:
            input_container_f.children = [calc_type_f_inv, prob_f]
        calcular_f()
    finally:
        _updating_f = False

# Registrar observadores apenas uma vez
if not hasattr(modo_f, '_handler_registered'):
    modo_f.observe(atualizar_interface_f, 'value')
    calc_type_f.observe(atualizar_interface_f, 'value')
    dfn_f.observe(calcular_f, 'value')
    dfd_f.observe(calcular_f, 'value')
    x_f.observe(calcular_f, 'value')
    a_f.observe(calcular_f, 'value')
    b_f.observe(calcular_f, 'value')
    prob_f.observe(calcular_f, 'value')
    calc_type_f_inv.observe(calcular_f, 'value')
    show_graph_f.observe(calcular_f, 'value')
    
    modo_f._handler_registered = True
    calc_type_f._handler_registered = True
    dfn_f._handler_registered = True
    dfd_f._handler_registered = True
    x_f._handler_registered = True
    a_f._handler_registered = True
    b_f._handler_registered = True
    prob_f._handler_registered = True
    calc_type_f_inv._handler_registered = True
    show_graph_f._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora F (Fisher-Snedecor)</h3>"),
    modo_f,
    dfn_f, 
    dfd_f, 
    input_container_f,
    show_graph_f, 
    output_f
]))

atualizar_interface_f()

## 5. Distribuição Exponencial

A distribuição exponencial modela o tempo entre eventos em um processo de Poisson.

**Parâmetros:**
- λ (lambda): taxa de ocorrência
- X: tempo

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_exp = False

# Criar widgets
modo_exp = widgets.ToggleButtons(
    options=['Direto (x → P)', 'Inverso (P → x)'],
    value='Direto (x → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

lambda_exp = widgets.FloatText(value=0.5, description='λ (taxa):', style={'description_width': 'initial'})
x_exp = widgets.FloatText(value=2.0, description='x (tempo):', style={'description_width': 'initial'})
a_exp = widgets.FloatText(value=1.0, description='a (limite inf.):', style={'description_width': 'initial'})
b_exp = widgets.FloatText(value=4.0, description='b (limite sup.):', style={'description_width': 'initial'})
prob_exp = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_exp = widgets.Dropdown(
    options=[
        ('P(X ≤ x)', 'cdf'), 
        ('P(X ≥ x)', 'sf'), 
        ('P(X > x)', 'gt'), 
        ('P(a ≤ X ≤ b)', 'interval'),
        ('f(x) (PDF)', 'pdf')
    ],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_exp_inv = widgets.Dropdown(
    options=[('P(X ≤ x) = p', 'cdf'), ('P(X ≥ x) = p', 'sf')],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph_exp = widgets.Checkbox(value=True, description='Mostrar gráfico')
output_exp = widgets.Output()
input_container_exp = widgets.VBox()

def calcular_exponencial(change=None):
    global _updating_exp
    if _updating_exp:
        return
    _updating_exp = True
    
    try:
        with output_exp:
            clear_output(wait=True)
            try:
                lam = lambda_exp.value
                
                if lam <= 0:
                    print("❌ λ deve ser > 0")
                    return
                
                dist = stats.expon(scale=1/lam)
                
                # Modo Direto
                if modo_exp.value == 'Direto (x → P)':
                    if calc_type_exp.value == 'interval':
                        a = a_exp.value
                        b = b_exp.value
                        
                        if a < 0 or b < 0:
                            print("❌ a e b devem ser ≥ 0")
                            return
                        
                        if a >= b:
                            print("❌ a deve ser menor que b")
                            return
                        
                        prob = dist.cdf(b) - dist.cdf(a)
                        print(f"P({a:.4f} ≤ X ≤ {b:.4f}) = {prob:.6f}")
                        print(f"\nLimite inferior: a = {a:.4f}")
                        print(f"Limite superior: b = {b:.4f}")
                        print(f"Amplitude: {b-a:.4f}")
                        
                        if show_graph_exp.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(0, 5/lam, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='teal', linewidth=2, label='PDF')
                            ax.axvline(a, color='red', linestyle='--', linewidth=2, label=f'a = {a:.2f}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b:.2f}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            x_fill = x_vals[(x_vals >= a) & (x_vals <= b)]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='teal', label=f'P = {prob:.4f}')
                            
                            ax.set_xlabel('x (tempo)', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Exponencial: λ={lam}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                    else:
                        x = x_exp.value
                        
                        if x < 0:
                            print("❌ x deve ser ≥ 0")
                            return
                        
                        if calc_type_exp.value == 'pdf':
                            result = dist.pdf(x)
                            print(f"f({x:.4f}) = {result:.6f}")
                        elif calc_type_exp.value == 'cdf':
                            result = dist.cdf(x)
                            print(f"P(X ≤ {x:.4f}) = {result:.6f}")
                        elif calc_type_exp.value == 'sf':
                            result = dist.sf(x)
                            print(f"P(X ≥ {x:.4f}) = {result:.6f}")
                        elif calc_type_exp.value == 'gt':
                            result = dist.sf(x)
                            print(f"P(X > {x:.4f}) = {result:.6f}")
                        
                        print(f"\nTaxa λ = {lam:.4f}")
                        print(f"Média E(X) = {dist.mean():.4f}")
                        print(f"Variância Var(X) = {dist.var():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        
                        if show_graph_exp.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(0, 5/lam, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='teal', linewidth=2, label='PDF')
                            ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f}')
                            ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            if calc_type_exp.value == 'cdf':
                                x_fill = x_vals[x_vals <= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='teal')
                            elif calc_type_exp.value in ['sf', 'gt']:
                                x_fill = x_vals[x_vals >= x]
                                ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='teal')
                            
                            ax.set_xlabel('x (tempo)', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Exponencial: λ={lam}', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    p = prob_exp.value
                    
                    if p <= 0 or p >= 1:
                        print("❌ Probabilidade deve estar entre 0 e 1 (exclusivo)")
                        return
                    
                    if calc_type_exp_inv.value == 'cdf':
                        x = dist.ppf(p)
                        print(f"Para P(X ≤ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    else:  # sf
                        x = dist.isf(p)
                        print(f"Para P(X ≥ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    
                    print(f"\nTaxa λ = {lam:.4f}")
                    print(f"Média E(X) = {dist.mean():.4f}")
                    print(f"Variância Var(X) = {dist.var():.4f}")
                    
                    if show_graph_exp.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.linspace(0, 5/lam, 1000)
                        pdf_vals = dist.pdf(x_vals)
                        
                        ax.plot(x_vals, pdf_vals, color='teal', linewidth=2, label='PDF')
                        ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f} (quantil)')
                        ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        if calc_type_exp_inv.value == 'cdf':
                            x_fill = x_vals[x_vals <= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='teal', label=f'Área = {p:.4f}')
                        else:
                            x_fill = x_vals[x_vals >= x]
                            ax.fill_between(x_fill, dist.pdf(x_fill), alpha=0.3, color='teal', label=f'Área = {p:.4f}')
                        
                        ax.set_xlabel('x (tempo)', fontsize=12)
                        ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                        ax.set_title(f'Distribuição Exponencial: λ={lam}', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_exp = False

def atualizar_interface_exp(change=None):
    global _updating_exp
    if _updating_exp:
        return
    _updating_exp = True
    
    try:
        if modo_exp.value == 'Direto (x → P)':
            if calc_type_exp.value == 'interval':
                input_container_exp.children = [calc_type_exp, a_exp, b_exp]
            else:
                input_container_exp.children = [calc_type_exp, x_exp]
        else:
            input_container_exp.children = [calc_type_exp_inv, prob_exp]
        calcular_exponencial()
    finally:
        _updating_exp = False

# Registrar observadores apenas uma vez
if not hasattr(modo_exp, '_handler_registered'):
    modo_exp.observe(atualizar_interface_exp, 'value')
    calc_type_exp.observe(atualizar_interface_exp, 'value')
    lambda_exp.observe(calcular_exponencial, 'value')
    x_exp.observe(calcular_exponencial, 'value')
    a_exp.observe(calcular_exponencial, 'value')
    b_exp.observe(calcular_exponencial, 'value')
    prob_exp.observe(calcular_exponencial, 'value')
    calc_type_exp_inv.observe(calcular_exponencial, 'value')
    show_graph_exp.observe(calcular_exponencial, 'value')
    
    modo_exp._handler_registered = True
    calc_type_exp._handler_registered = True
    lambda_exp._handler_registered = True
    x_exp._handler_registered = True
    a_exp._handler_registered = True
    b_exp._handler_registered = True
    prob_exp._handler_registered = True
    calc_type_exp_inv._handler_registered = True
    show_graph_exp._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Exponencial</h3>"),
    modo_exp,
    lambda_exp, 
    input_container_exp,
    show_graph_exp, 
    output_exp
]))

atualizar_interface_exp()

## 6. Distribuição Uniforme Contínua

A distribuição uniforme contínua modela uma variável aleatória com probabilidade constante em um intervalo.

**Parâmetros:**
- a: limite inferior
- b: limite superior
- X: valor da variável

In [ ]:
# Flag para prevenir chamadas duplicadas
_updating_unif = False

# Criar widgets
modo_unif = widgets.ToggleButtons(
    options=['Direto (x → P)', 'Inverso (P → x)'],
    value='Direto (x → P)',
    description='Modo:',
    style={'description_width': 'initial', 'button_width': '150px'}
)

a_unif = widgets.FloatText(value=0, description='a (mín):', style={'description_width': 'initial'})
b_unif = widgets.FloatText(value=10, description='b (máx):', style={'description_width': 'initial'})
x_unif = widgets.FloatText(value=5.0, description='x (valor):', style={'description_width': 'initial'})
a_unif_int = widgets.FloatText(value=2.0, description='a (limite inf.):', style={'description_width': 'initial'})
b_unif_int = widgets.FloatText(value=8.0, description='b (limite sup.):', style={'description_width': 'initial'})
prob_unif = widgets.FloatText(value=0.5, description='Probabilidade:', style={'description_width': 'initial'})

calc_type_unif = widgets.Dropdown(
    options=[
        ('P(X ≤ x)', 'cdf'), 
        ('P(X ≥ x)', 'sf'), 
        ('P(a ≤ X ≤ b)', 'interval'),
        ('f(x) (PDF)', 'pdf')
    ],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

calc_type_unif_inv = widgets.Dropdown(
    options=[('P(X ≤ x) = p', 'cdf'), ('P(X ≥ x) = p', 'sf')],
    value='cdf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph_unif = widgets.Checkbox(value=True, description='Mostrar gráfico')
output_unif = widgets.Output()
input_container_unif = widgets.VBox()

def calcular_uniforme(change=None):
    global _updating_unif
    if _updating_unif:
        return
    _updating_unif = True
    
    try:
        with output_unif:
            clear_output(wait=True)
            try:
                a_min = a_unif.value
                b_max = b_unif.value
                
                if b_max <= a_min:
                    print("❌ b deve ser maior que a")
                    return
                
                dist = stats.uniform(loc=a_min, scale=b_max-a_min)
                
                # Modo Direto
                if modo_unif.value == 'Direto (x → P)':
                    if calc_type_unif.value == 'interval':
                        a = a_unif_int.value
                        b = b_unif_int.value
                        
                        if a >= b:
                            print("❌ a deve ser menor que b")
                            return
                        
                        prob = dist.cdf(b) - dist.cdf(a)
                        print(f"P({a:.4f} ≤ X ≤ {b:.4f}) = {prob:.6f}")
                        print(f"\nLimite inferior: a = {a:.4f}")
                        print(f"Limite superior: b = {b:.4f}")
                        print(f"Amplitude: {b-a:.4f}")
                        
                        if show_graph_unif.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(a_min - (b_max-a_min)*0.2, b_max + (b_max-a_min)*0.2, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='darkgreen', linewidth=2, label='PDF')
                            ax.axvline(a, color='red', linestyle='--', linewidth=2, label=f'a = {a:.2f}')
                            ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b:.2f}')
                            ax.axvline(dist.mean(), color='orange', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            if a_min <= a <= b_max and a_min <= b <= b_max:
                                x_fill = np.linspace(max(a, a_min), min(b, b_max), 100)
                                ax.fill_between(x_fill, 1/(b_max-a_min), alpha=0.3, color='green', label=f'P = {prob:.4f}')
                            
                            ax.set_xlabel('x', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Uniforme: U({a_min}, {b_max})', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            ax.set_ylim(bottom=0)
                            plt.tight_layout()
                            plt.show()
                    else:
                        x = x_unif.value
                        
                        if calc_type_unif.value == 'pdf':
                            result = dist.pdf(x)
                            print(f"f({x:.4f}) = {result:.6f}")
                            if x < a_min or x > b_max:
                                print(f"(x fora do intervalo [{a_min}, {b_max}])")
                        elif calc_type_unif.value == 'cdf':
                            result = dist.cdf(x)
                            print(f"P(X ≤ {x:.4f}) = {result:.6f}")
                        elif calc_type_unif.value == 'sf':
                            result = dist.sf(x)
                            print(f"P(X ≥ {x:.4f}) = {result:.6f}")
                        
                        print(f"\nIntervalo: [{a_min}, {b_max}]")
                        print(f"Média E(X) = {dist.mean():.4f}")
                        print(f"Variância Var(X) = {dist.var():.4f}")
                        print(f"Desvio Padrão σ = {dist.std():.4f}")
                        
                        if show_graph_unif.value:
                            fig, ax = plt.subplots(figsize=(10, 5))
                            x_vals = np.linspace(a_min - (b_max-a_min)*0.2, b_max + (b_max-a_min)*0.2, 1000)
                            pdf_vals = dist.pdf(x_vals)
                            
                            ax.plot(x_vals, pdf_vals, color='darkgreen', linewidth=2, label='PDF')
                            ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f}')
                            ax.axvline(dist.mean(), color='orange', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                            
                            if a_min <= x <= b_max:
                                if calc_type_unif.value == 'cdf':
                                    x_fill = np.linspace(a_min, min(x, b_max), 100)
                                    ax.fill_between(x_fill, 1/(b_max-a_min), alpha=0.3, color='green')
                                elif calc_type_unif.value == 'sf':
                                    x_fill = np.linspace(max(x, a_min), b_max, 100)
                                    ax.fill_between(x_fill, 1/(b_max-a_min), alpha=0.3, color='green')
                            
                            ax.set_xlabel('x', fontsize=12)
                            ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                            ax.set_title(f'Distribuição Uniforme: U({a_min}, {b_max})', fontsize=14, fontweight='bold')
                            ax.legend()
                            ax.grid(True, alpha=0.3)
                            ax.set_ylim(bottom=0)
                            plt.tight_layout()
                            plt.show()
                
                # Modo Inverso
                else:
                    p = prob_unif.value
                    
                    if p <= 0 or p >= 1:
                        print("❌ Probabilidade deve estar entre 0 e 1 (exclusivo)")
                        return
                    
                    if calc_type_unif_inv.value == 'cdf':
                        x = dist.ppf(p)
                        print(f"Para P(X ≤ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    else:  # sf
                        x = dist.isf(p)
                        print(f"Para P(X ≥ x) = {p:.6f}")
                        print(f"x = {x:.4f}")
                    
                    print(f"\nIntervalo: [{a_min}, {b_max}]")
                    print(f"Média E(X) = {dist.mean():.4f}")
                    print(f"Variância Var(X) = {dist.var():.4f}")
                    
                    if show_graph_unif.value:
                        fig, ax = plt.subplots(figsize=(10, 5))
                        x_vals = np.linspace(a_min - (b_max-a_min)*0.2, b_max + (b_max-a_min)*0.2, 1000)
                        pdf_vals = dist.pdf(x_vals)
                        
                        ax.plot(x_vals, pdf_vals, color='darkgreen', linewidth=2, label='PDF')
                        ax.axvline(x, color='red', linestyle='--', linewidth=2, label=f'x = {x:.2f} (quantil)')
                        ax.axvline(dist.mean(), color='orange', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                        
                        if calc_type_unif_inv.value == 'cdf':
                            x_fill = np.linspace(a_min, x, 100)
                            ax.fill_between(x_fill, 1/(b_max-a_min), alpha=0.3, color='green', label=f'Área = {p:.4f}')
                        else:
                            x_fill = np.linspace(x, b_max, 100)
                            ax.fill_between(x_fill, 1/(b_max-a_min), alpha=0.3, color='green', label=f'Área = {p:.4f}')
                        
                        ax.set_xlabel('x', fontsize=12)
                        ax.set_ylabel('Densidade de Probabilidade f(x)', fontsize=12)
                        ax.set_title(f'Distribuição Uniforme: U({a_min}, {b_max})', fontsize=14, fontweight='bold')
                        ax.legend()
                        ax.grid(True, alpha=0.3)
                        ax.set_ylim(bottom=0)
                        plt.tight_layout()
                        plt.show()
                
            except Exception as e:
                print(f"❌ Erro: {str(e)}")
    finally:
        _updating_unif = False

def atualizar_interface_unif(change=None):
    global _updating_unif
    if _updating_unif:
        return
    _updating_unif = True
    
    try:
        if modo_unif.value == 'Direto (x → P)':
            if calc_type_unif.value == 'interval':
                input_container_unif.children = [calc_type_unif, a_unif_int, b_unif_int]
            else:
                input_container_unif.children = [calc_type_unif, x_unif]
        else:
            input_container_unif.children = [calc_type_unif_inv, prob_unif]
        calcular_uniforme()
    finally:
        _updating_unif = False

# Registrar observadores apenas uma vez
if not hasattr(modo_unif, '_handler_registered'):
    modo_unif.observe(atualizar_interface_unif, 'value')
    calc_type_unif.observe(atualizar_interface_unif, 'value')
    a_unif.observe(calcular_uniforme, 'value')
    b_unif.observe(calcular_uniforme, 'value')
    x_unif.observe(calcular_uniforme, 'value')
    a_unif_int.observe(calcular_uniforme, 'value')
    b_unif_int.observe(calcular_uniforme, 'value')
    prob_unif.observe(calcular_uniforme, 'value')
    calc_type_unif_inv.observe(calcular_uniforme, 'value')
    show_graph_unif.observe(calcular_uniforme, 'value')
    
    modo_unif._handler_registered = True
    calc_type_unif._handler_registered = True
    a_unif._handler_registered = True
    b_unif._handler_registered = True
    x_unif._handler_registered = True
    a_unif_int._handler_registered = True
    b_unif_int._handler_registered = True
    prob_unif._handler_registered = True
    calc_type_unif_inv._handler_registered = True
    show_graph_unif._handler_registered = True

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Uniforme Contínua</h3>"),
    modo_unif,
    a_unif, 
    b_unif, 
    input_container_unif,
    show_graph_unif, 
    output_unif
]))

atualizar_interface_unif()